# Experiment 2: Zero-Shot Benchmark on Credit Data

Runs every model on raw, unprocessed data and produces a comprehensive comparison.

**What this experiment measures:**
- No preprocessing, no feature engineering, no hyperparameter tuning
- True out-of-the-box capability of each model
- 7+ metrics including calibration (critical for credit scoring)
- Timing and memory usage

**Prerequisites:** `uv sync` or `pip install -e .`

In [1]:
import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_credit_dataset, get_dataset_info
from src.evaluation.metrics import compute_all_metrics
from src.visualization.plots import (
    plot_leaderboard, plot_time_vs_accuracy,
    plot_calibration_diagrams, plot_limitation_matrix, set_style,
)

RESULTS_DIR = Path('../results')
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
set_style()
print('Setup complete.')

Setup complete.


## 1. Load Dataset

Change `DATASET` to `'german_credit'` or `'taiwan_credit'` for smaller/faster runs.

In [2]:
DATASET = 'german_credit'  # Options: 'give_me_credit' | 'german_credit' | 'taiwan_credit'

X_train, X_test, y_train, y_test = load_credit_dataset(
    DATASET, test_size=0.2, random_state=RANDOM_STATE
)

info = get_dataset_info(DATASET)
print(f'Dataset       : {DATASET}')
print(f'Train shape   : {X_train.shape}')
print(f'Test shape    : {X_test.shape}')
print(f'Positive rate : {y_train.mean():.4f}')
print(f'Missing values: {X_train.isna().sum().sum()} '
      f'({X_train.isna().mean().mean()*100:.1f}%)')
print(f'\nFeature dtypes:')
print(X_train.dtypes.value_counts().to_string())

Dataset       : german_credit
Train shape   : (800, 20)
Test shape    : (200, 20)
Positive rate : 0.3000
Missing values: 0 (0.0%)

Feature dtypes:
object    13
int64      7


## 2. Load Model Registry

Models not installed are skipped gracefully with a warning.

In [3]:
from scripts.run_benchmark import get_zero_shot_models

models = get_zero_shot_models()

print(f'\n{len(models)} models loaded:\n')
print(f'  {"Status":<8} {"Model":<28} {"Max Rows":<12} {"License"}')
print(f'  {"-"*70}')
for m in models:
    lim = m.get_limitations()
    fits = m.can_handle_dataset(len(X_train), X_train.shape[1])
    status = 'OK' if fits else 'SKIP'
    max_r = str(lim.max_rows) if lim.max_rows else 'unlimited'
    print(f'  {status:<8} {m.name:<28} {max_r:<12} {lim.license[:35]}')


17 models loaded:

  Status   Model                        Max Rows     License
  ----------------------------------------------------------------------
  OK       TabPFN-v1                    1000         Apache 2.0
  OK       TabPFN-v2                    10000        Prior Labs License (commercial with
  OK       TabPFN-v2.5                  50000        TabPFN-2.5 License v1.0 (NON-COMMER
  OK       TabPFN-v2.5-real             50000        TabPFN-2.5 License v1.0 (NON-COMMER
  OK       TabICL-v2                    unlimited    BSD-3-Clause
  OK       TabICL-v1.1                  unlimited    BSD-3-Clause
  OK       Mitra                        unlimited    Apache 2.0
  OK       TabDPT                       unlimited    MIT
  OK       TabNet                       unlimited    Apache 2.0
  OK       FT-Transformer               unlimited    MIT
  OK       XGBoost-Default              unlimited    Apache 2.0
  OK       XGBoost-Tuned                unlimited    Apache 2.0
  OK       Ca

## 3. Run Zero-Shot Evaluation

Each model receives the exact same raw data. Results are saved after each model
so a crash does not lose previous work.

In [4]:
all_results = []
predictions = {}  # model_name -> (y_true, y_proba) for calibration plots

for model in models:
    print(f'\n[{model.name}]', end=' ', flush=True)

    result = model.evaluate(X_train, y_train, X_test, y_test, DATASET, 'zero_shot')
    all_results.append(result.to_dict())

    if result.success:
        print(
            f'AUC={result.auc_roc:.4f}  '
            f'LogLoss={result.log_loss_val:.4f}  '
            f'ECE={result.ece:.4f}  '
            f'Time={result.total_time:.1f}s'
        )
        # Store probabilities for calibration plot
        try:
            proba = model.predict_proba(X_test)
            proba = proba[:, 1] if proba.ndim > 1 else proba
            predictions[model.name] = (y_test.values, proba)
        except Exception:
            pass
    else:
        print(f'FAILED: {result.error_message[:80]}')

results_df = pd.DataFrame(all_results)
out_path = RESULTS_DIR / f'zero_shot_{DATASET}.csv'
results_df.to_csv(out_path, index=False)
print(f'\nResults saved to: {out_path}')


[TabPFN-v1] FAILED: ModuleNotFoundError: No module named 'tabpfn'

[TabPFN-v2] FAILED: ModuleNotFoundError: No module named 'tabpfn'

[TabPFN-v2.5] FAILED: ModuleNotFoundError: No module named 'tabpfn'

[TabPFN-v2.5-real] FAILED: ModuleNotFoundError: No module named 'tabpfn'

[TabICL-v2] FAILED: ModuleNotFoundError: No module named 'tabicl'

[TabICL-v1.1] FAILED: ModuleNotFoundError: No module named 'tabicl'

[Mitra] FAILED: ModuleNotFoundError: No module named 'autogluon'

[TabDPT] FAILED: ImportError: TabDPT not installed. Install with:
  pip install git+https://githu

[TabNet] FAILED: ModuleNotFoundError: No module named 'pytorch_tabnet'

[FT-Transformer] FAILED: ImportError: rtdl not installed. Install with:
  pip install rtdl
or: pip instal

[XGBoost-Default] FAILED: ModuleNotFoundError: No module named 'xgboost'

[XGBoost-Tuned] FAILED: ModuleNotFoundError: No module named 'xgboost'

[CatBoost-Default] FAILED: ModuleNotFoundError: No module named 'catboost'

[CatBoost-Tuned] FAI

## 4. Leaderboard

In [ ]:
successful = results_df[results_df['success'] == True].copy()

cols = ['model_name', 'auc_roc', 'log_loss_val', 'brier_score',
        'ece', 'f1_macro', 'accuracy', 'total_time', 'peak_memory_mb']
leaderboard = (
    successful[cols]
    .sort_values('auc_roc', ascending=False)
    .reset_index(drop=True)
)
leaderboard.index += 1
leaderboard.index.name = 'Rank'

fmt = {
    'auc_roc': '{:.4f}', 'log_loss_val': '{:.4f}', 'brier_score': '{:.4f}',
    'ece': '{:.4f}', 'f1_macro': '{:.4f}', 'accuracy': '{:.4f}',
    'total_time': '{:.2f}', 'peak_memory_mb': '{:.0f}',
}

try:
    # Styled table (requires jinja2)
    display(
        leaderboard.style
        .format(fmt)
        .background_gradient(subset=['auc_roc'], cmap='Greens')
        .background_gradient(subset=['log_loss_val'], cmap='Reds_r')
        .background_gradient(subset=['ece'], cmap='Reds_r')
    )
except Exception:
    # Plain fallback if jinja2 is missing
    display(leaderboard)

## 5. Visualisations

In [ ]:
# Bar chart leaderboard
if not successful.empty:
    plot_leaderboard(
        results_df,
        metric='auc_roc',
        title=f'Zero-Shot AUC-ROC — {DATASET.replace("_", " ").title()}',
        save_path=str(FIGURES_DIR / f'leaderboard_{DATASET}.png'),
    )
    plt.show()
else:
    print('No successful results to plot.')

In [ ]:
# Accuracy vs speed trade-off scatter plot
df_timed = successful[successful['total_time'].notna() & (successful['total_time'] > 0)]
if not df_timed.empty:
    plot_time_vs_accuracy(
        df_timed,
        metric='auc_roc',
        save_path=str(FIGURES_DIR / f'time_vs_accuracy_{DATASET}.png'),
    )
    plt.show()
else:
    print('No timing data available.')

In [ ]:
# Calibration (reliability) diagrams
if predictions:
    plot_calibration_diagrams(
        predictions,
        n_bins=10,
        save_path=str(FIGURES_DIR / f'calibration_{DATASET}.png'),
    )
    plt.show()
else:
    print('No predictions stored for calibration plot.')

## 6. Model Scalability Matrix

In [ ]:
models_info = []
for m in models:
    lim = m.get_limitations()
    models_info.append({
        'name': m.name,
        'max_rows': lim.max_rows,
        'recommended_max_rows': lim.recommended_max_rows,
        'license': lim.license,
        'commercial': lim.commercial_use,
    })

plot_limitation_matrix(
    models_info,
    save_path=str(FIGURES_DIR / 'limitation_matrix.png'),
)
plt.show()

## 7. License Summary

In [ ]:
print(f'  {"Model":<28} {"Commercial?":<20} License')
print(f'  {"-"*70}')
for m in models_info:
    tag = 'YES' if m['commercial'] else 'NON-COMMERCIAL ONLY'
    print(f'  {m["name"]:<28} {tag:<20} {m["license"]}')

## 8. Failed Models Analysis

In [ ]:
failed = results_df[results_df['success'] == False]
if not failed.empty:
    print(f'{len(failed)} model(s) failed or were skipped:\n')
    for _, row in failed.iterrows():
        print(f'  {row["model_name"]}')
        print(f'    {row["error_message"]}')
        print()
else:
    print('All models succeeded!')

---
## Summary Checklist

After running this notebook, answer:
1. Which TFMs beat XGBoost-Default in AUC-ROC?
2. Which TFMs match XGBoost-Tuned?
3. Are TFM probabilities better calibrated (lower ECE)?
4. What is the speed/accuracy Pareto frontier?
5. Which models failed due to row limits vs. missing packages?

Next: `04_scaling_experiments.ipynb` — tests each model at increasing dataset sizes.